In [ ]:
# openclawqwen.ipynb single-cell launcher for Colab (T4 12GB)
import os, sys, time, json, shutil, signal, subprocess, threading
from pathlib import Path

print("🚀 Bootstrapping: vLLM + OpenClaw + Qwen in one Colab cell")

# ---------------------------
# Config (easy to edit)
# ---------------------------
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")  # Replace with Qwen3.5-9B HF id when available
VLLM_PORT = int(os.environ.get("VLLM_PORT", "8000"))
BASE_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
HOST = "127.0.0.1"
DTYPE = os.environ.get("DTYPE", "float16")
MAX_MODEL_LEN = int(os.environ.get("MAX_MODEL_LEN", "4096"))  # safer default for T4 12GB
GPU_MEMORY_UTILIZATION = float(os.environ.get("GPU_MEMORY_UTILIZATION", "0.86"))
TEMPERATURE = float(os.environ.get("TEMPERATURE", "0.6"))
TOP_P = float(os.environ.get("TOP_P", "0.9"))
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "512"))
OPENCLAW_REPO = os.environ.get("OPENCLAW_REPO", "https://github.com/openclaw/openclaw.git")
OPENCLAW_DIR = Path("/content/openclaw")

print(f"📦 MODEL_ID={MODEL_ID}")
print(f"🔌 BASE_URL={BASE_URL}")

# ---------------------------
# Helpers
# ---------------------------
def run(cmd, check=True, env=None):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check, env=env)


def pip_install(packages):
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
    run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)


def wait_for_vllm(timeout=1200):
    import requests
    t0 = time.time()
    last_error = None
    while time.time() - t0 < timeout:
        try:
            r = requests.get(f"http://{HOST}:{VLLM_PORT}/v1/models", timeout=5)
            if r.ok:
                print("✅ vLLM is up and responding at /v1/models")
                return True
            last_error = f"HTTP {r.status_code}: {r.text[:200]}"
        except Exception as e:
            last_error = str(e)
        elapsed = int(time.time() - t0)
        if elapsed % 30 == 0:
            print(f"⏳ Waiting for vLLM startup... {elapsed}s")
        time.sleep(2)
    print("❌ vLLM did not start in time.")
    if last_error:
        print("Last error:", last_error)
    return False

# ---------------------------
# Step 1: install deps
# ---------------------------
print("\n[1/5] Installing dependencies...")
pip_install([
    "vllm>=0.8.0",
    "gradio>=5.0.0",
    "openai>=1.40.0",
    "requests>=2.31.0",
    "httpx>=0.27.0",
    "nest_asyncio>=1.6.0",
])

# Try to install OpenClaw (best-effort, with graceful fallback)
openclaw_ready = False
print("\n[2/5] Preparing OpenClaw...")
try:
    if OPENCLAW_DIR.exists():
        shutil.rmtree(OPENCLAW_DIR)
    run(["git", "clone", "--depth", "1", OPENCLAW_REPO, str(OPENCLAW_DIR)], check=True)
    pyproject = OPENCLAW_DIR / "pyproject.toml"
    reqs = OPENCLAW_DIR / "requirements.txt"
    if pyproject.exists() or reqs.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "-e", str(OPENCLAW_DIR)], check=False)
    # detect import or CLI
    cli_exists = shutil.which("openclaw") is not None
    try:
        __import__("openclaw")
        openclaw_ready = True
    except Exception:
        openclaw_ready = cli_exists
except Exception as e:
    print(f"⚠️ OpenClaw setup warning: {e}")

if openclaw_ready:
    print("✅ OpenClaw package/CLI detected.")
else:
    print("⚠️ OpenClaw runtime was not detected; using compatible agent loop fallback over OpenAI API.")

# ---------------------------
# Step 3: start vLLM server
# ---------------------------
print("\n[3/5] Starting vLLM OpenAI-compatible server...")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--host", HOST,
    "--port", str(VLLM_PORT),
    "--model", MODEL_ID,
    "--dtype", DTYPE,
    "--tensor-parallel-size", "1",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--trust-remote-code",
    "--disable-log-requests",
    "--max-num-seqs", "4",
]

vllm_log = open("/content/vllm.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

if not wait_for_vllm(timeout=1800):
    print("\n--- Last vLLM logs ---")
    try:
        with open("/content/vllm.log", "r") as f:
            print(f.read()[-6000:])
    except Exception:
        pass
    raise RuntimeError("vLLM failed to start; inspect /content/vllm.log")

# ---------------------------
# Step 4: connect agent layer
# ---------------------------
print("\n[4/5] Connecting agent layer...")
os.environ["OPENAI_API_BASE"] = BASE_URL
os.environ["OPENAI_BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "EMPTY")
os.environ["OPENCLAW_BASE_URL"] = BASE_URL
os.environ["OPENCLAW_MODEL"] = MODEL_ID

from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=os.environ["OPENAI_API_KEY"])

# Minimal tool set for agent behavior
import datetime

def tool_now_utc(_: str = ""):
    return {"utc": datetime.datetime.utcnow().isoformat() + "Z"}

def tool_calc(expression: str):
    allowed = set("0123456789+-*/(). %")
    if not expression or any(c not in allowed for c in expression):
        return {"error": "Only arithmetic expressions are allowed."}
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "now_utc",
            "description": "Return current UTC time.",
            "parameters": {"type": "object", "properties": {"dummy": {"type": "string"}}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a simple arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
]


def run_agent(messages, max_loops=4):
    # OpenClaw-like agent loop via tool-calling over OpenAI-compatible backend
    local_messages = list(messages)
    for _ in range(max_loops):
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=local_messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
        )
        msg = resp.choices[0].message
        if not getattr(msg, "tool_calls", None):
            return msg.content or ""

        local_messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            if name == "now_utc":
                result = tool_now_utc(args.get("dummy", ""))
            elif name == "calculator":
                result = tool_calc(args.get("expression", ""))
            else:
                result = {"error": f"Unknown tool {name}"}
            local_messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": name,
                "content": json.dumps(result, ensure_ascii=False),
            })
    return "Agent loop limit reached."

print("✅ Agent backend connected to local vLLM endpoint.")
if openclaw_ready:
    print("✅ OpenClaw detected. (This notebook uses a minimal OpenClaw-compatible tool loop for Colab reliability.)")
else:
    print("ℹ️ Running fallback OpenClaw-compatible loop (tools + function calling).")

# ---------------------------
# Step 5: Gradio UI
# ---------------------------
print("\n[5/5] Launching Gradio chat UI...")
import gradio as gr

SYSTEM_PROMPT = (
    "You are an OpenClaw-style autonomous assistant. "
    "Use tools when needed, keep answers concise and actionable."
)


def chat_fn(user_message, history):
    history = history or []
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for u, a in history:
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": user_message})
    answer = run_agent(messages)
    return answer

with gr.Blocks(title="OpenClaw + vLLM + Qwen (Colab)") as demo:
    gr.Markdown("## 🦞 OpenClaw + vLLM + Qwen chat\nIf the public link is shown below, open it to chat.")
    chatbot = gr.Chatbot(height=500)
    msg = gr.Textbox(label="Message", placeholder="Ask anything…")
    clear = gr.Button("Clear")

    def user_submit(user_message, history):
        answer = chat_fn(user_message, history)
        history = history + [(user_message, answer)]
        return "", history

    msg.submit(user_submit, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)

print("🎉 Stack is ready: model loading done/ongoing in vLLM, agent connected, UI starting.")
demo.launch(share=True, debug=False, show_error=True)
